In [8]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score

In [9]:
splits = joblib.load("splits_cache_delta_only.pkl")

In [10]:
splits[0]['X_train_raw'].shape

(920, 152)

In [11]:
features = []

for split in splits:
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    feature_name_coef = list(zip(X_train.columns, model.coef_[0]))
    feature_name_coef.sort(key=lambda x: abs(x[1]), reverse=True)
    features.append([name for name, coef in feature_name_coef[:30]])

In [12]:
print(features)

[['delta_map_score_delta_sum_6', 'delta_player_map_stats_specific_kills_median_2', 'delta_player_map_stats_specific_kill_death_differential_median_2', 'delta_map_score_delta_sum_4', 'delta_map_played', 'delta_match_patch_score_delta_sum', 'delta_player_map_stats_specific_clutch_sum', 'delta_map_score_delta_avg', 'delta_player_map_stats_specific_kill_assist_survive_trade_ratio_median', 'delta_player_map_stats_specific_headshot_percentage_median', 'delta_player_map_stats_specific_kill_death_differential_median', 'delta_player_map_stats_specific_average_damage_per_round_median', 'delta_map_patch_played', 'delta_player_map_stats_specific_first_kill_first_death_differential_median_4', 'delta_player_map_stats_specific_econ_rating_median_2', 'delta_match_best_of_sum', 'delta_match_score_delta_sum_2', 'delta_player_map_stats_specific_kills_median_6', 'delta_player_map_stats_specific_assists_median', 'delta_player_map_stats_specific_first_kill_first_death_differential_median_2', 'delta_player_m

In [13]:
def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi

In [14]:
def make_objective(X_train, y_train, X_val, y_val):
    def objective(trial):
        C = trial.suggest_float('C', 1e-5, 10, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1)

        all_val_true, all_val_proba = [], []

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled   = scaler.transform(X_val)

        selector = VarianceThreshold(threshold=0.01)
        X_train_sel = selector.fit_transform(X_train_scaled)
        X_val_sel   = selector.transform(X_val_scaled)

        model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver='saga', max_iter=5000)
        model.fit(X_train_sel, y_train)

        val_proba = model.predict_proba(X_val_sel)[:, 1]
        all_val_true.append(y_val); all_val_proba.append(val_proba)
        logloss = log_loss(all_val_true, all_val_proba)
        return logloss

    return objective

In [15]:
from sklearn.preprocessing import PolynomialFeatures


scores = []
parameters = []

polynomial_features = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

for split, feature_list in zip(splits, features):
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    X_train = X_train[feature_list]
    X_val = X_val[feature_list]

    X_train = polynomial_features.fit_transform(X_train)
    X_val = polynomial_features.transform(X_val)

    objective = make_objective(X_train, y_train, X_val, y_val)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=1000, n_jobs=-1)
    params = study.best_params
    parameters.append(params)
    print("Best parameters:", params)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_pred = model.predict(X_val_sel)
    scores.append(roc_auc_score(y_val, val_pred))

print(f"Scores: {scores}")
print(f"Parameters: {parameters}")
print(f"Mean logloss: {np.mean(scores)}")

[I 2026-07-12 20:46:51,894] A new study created in memory with name: no-name-702adf38-975a-411f-80de-70258d7d9f3f
[I 2026-07-12 20:46:51,993] Trial 6 finished with value: 69.01539486046602 and parameters: {'C': 3.4841864480505336e-05, 'l1_ratio': 0.930074667952625}. Best is trial 6 with value: 69.01539486046602.
[I 2026-07-12 20:46:52,002] Trial 2 finished with value: 68.69193267367336 and parameters: {'C': 0.00010975439284027804, 'l1_ratio': 0.22330982473216332}. Best is trial 2 with value: 68.69193267367336.
[I 2026-07-12 20:46:52,034] Trial 0 finished with value: 68.32516593003942 and parameters: {'C': 0.006545730821233032, 'l1_ratio': 0.9669841108661653}. Best is trial 0 with value: 68.32516593003942.
[I 2026-07-12 20:46:52,039] Trial 10 finished with value: 68.16373987099445 and parameters: {'C': 0.000413149090682836, 'l1_ratio': 0.9652416823241997}. Best is trial 10 with value: 68.16373987099445.
[I 2026-07-12 20:46:52,041] Trial 9 finished with value: 68.16953668591148 and param

Best parameters: {'C': 0.013564583141963325, 'l1_ratio': 0.175026644135445}


[I 2026-07-12 20:50:11,674] A new study created in memory with name: no-name-72dd9adb-4999-400f-aebe-6667c8a0b07a
[I 2026-07-12 20:50:11,814] Trial 3 finished with value: 45.81616182035304 and parameters: {'C': 1.1413912670236763e-05, 'l1_ratio': 0.8812830554705647}. Best is trial 3 with value: 45.81616182035304.
[I 2026-07-12 20:50:11,826] Trial 8 finished with value: 45.76733932768868 and parameters: {'C': 0.0004779543786338412, 'l1_ratio': 0.5437345030414068}. Best is trial 8 with value: 45.76733932768868.
[I 2026-07-12 20:50:11,830] Trial 6 finished with value: 45.46184457511033 and parameters: {'C': 0.0003655867758464069, 'l1_ratio': 0.5626547133267832}. Best is trial 6 with value: 45.46184457511033.
[I 2026-07-12 20:50:11,831] Trial 0 finished with value: 46.004886823355264 and parameters: {'C': 0.00018562807177455922, 'l1_ratio': 0.42724337507547727}. Best is trial 6 with value: 45.46184457511033.
[I 2026-07-12 20:50:11,846] Trial 9 finished with value: 45.621561482972886 and pa

Best parameters: {'C': 0.047045268095773064, 'l1_ratio': 8.480545936004075e-05}


[I 2026-07-12 20:57:00,111] A new study created in memory with name: no-name-f93c67fc-41a5-431b-8d09-f468877950e7
[I 2026-07-12 20:57:00,289] Trial 0 finished with value: 86.61489306836792 and parameters: {'C': 2.0479490383605994e-05, 'l1_ratio': 0.8403745086530316}. Best is trial 0 with value: 86.61489306836792.
[I 2026-07-12 20:57:00,308] Trial 3 finished with value: 86.7354401284553 and parameters: {'C': 4.456077149758339e-05, 'l1_ratio': 0.8577645607581081}. Best is trial 0 with value: 86.61489306836792.
[I 2026-07-12 20:57:00,319] Trial 12 finished with value: 86.5395730331382 and parameters: {'C': 1.1224121108821734e-05, 'l1_ratio': 0.6074859809847719}. Best is trial 12 with value: 86.5395730331382.
[I 2026-07-12 20:57:00,326] Trial 11 finished with value: 86.5681293338391 and parameters: {'C': 0.0002579471603999945, 'l1_ratio': 0.4358103948698543}. Best is trial 12 with value: 86.5395730331382.
[I 2026-07-12 20:57:00,376] Trial 6 finished with value: 86.60023116108003 and parame

...

Best parameters: {'C': 0.005876068190301071, 'l1_ratio': 0.042986410570935545}


[I 2026-07-12 21:02:53,263] A new study created in memory with name: no-name-0a5ca8d0-e497-4ee7-a91c-58994ece16d3
[I 2026-07-12 21:02:53,449] Trial 12 finished with value: 110.8315458058944 and parameters: {'C': 4.447839249887433e-05, 'l1_ratio': 0.6165342856380659}. Best is trial 12 with value: 110.8315458058944.
[I 2026-07-12 21:02:53,466] Trial 15 finished with value: 110.78247630266686 and parameters: {'C': 2.767266021592639e-05, 'l1_ratio': 0.7871883892903743}. Best is trial 15 with value: 110.78247630266686.
[I 2026-07-12 21:02:53,469] Trial 13 finished with value: 110.29138238998894 and parameters: {'C': 0.00033226025953383135, 'l1_ratio': 0.423153936680878}. Best is trial 13 with value: 110.29138238998894.
[I 2026-07-12 21:02:53,480] Trial 2 finished with value: 111.05805974327154 and parameters: {'C': 0.001154573371701429, 'l1_ratio': 0.44235039814160215}. Best is trial 13 with value: 110.29138238998894.
[I 2026-07-12 21:02:53,483] Trial 14 finished with value: 111.03524202412

Best parameters: {'C': 0.05561754191984014, 'l1_ratio': 0.8672488159736376}


[I 2026-07-12 21:06:40,186] A new study created in memory with name: no-name-b551ae45-64e7-4503-a2d2-1353f400d283
[I 2026-07-12 21:06:40,425] Trial 2 finished with value: 40.44169694684759 and parameters: {'C': 5.453928228274523e-05, 'l1_ratio': 0.9077458459333859}. Best is trial 2 with value: 40.44169694684759.
[I 2026-07-12 21:06:40,441] Trial 9 finished with value: 40.324571216710865 and parameters: {'C': 0.0003914852295607506, 'l1_ratio': 0.5515184716838192}. Best is trial 9 with value: 40.324571216710865.
[I 2026-07-12 21:06:40,449] Trial 3 finished with value: 40.1828651189086 and parameters: {'C': 2.364221496814658e-05, 'l1_ratio': 0.12074079316914821}. Best is trial 3 with value: 40.1828651189086.
[I 2026-07-12 21:06:40,453] Trial 4 finished with value: 40.01007866682051 and parameters: {'C': 3.6851693518031445e-05, 'l1_ratio': 0.5985495862456468}. Best is trial 4 with value: 40.01007866682051.
[I 2026-07-12 21:06:40,456] Trial 11 finished with value: 40.32162624229514 and para

Best parameters: {'C': 9.994663647204307, 'l1_ratio': 9.974085680608579e-05}
Scores: [0.6363636363636362, 0.5606060606060606, 0.6719999999999999, 0.6746031746031746, 0.6625, 0.6896551724137931]
Parameters: [{'C': 0.013564583141963325, 'l1_ratio': 0.175026644135445}, {'C': 0.047045268095773064, 'l1_ratio': 8.480545936004075e-05}, {'C': 0.06023660679205369, 'l1_ratio': 0.9985421623225007}, {'C': 0.005876068190301071, 'l1_ratio': 0.042986410570935545}, {'C': 0.05561754191984014, 'l1_ratio': 0.8672488159736376}, {'C': 9.994663647204307, 'l1_ratio': 9.974085680608579e-05}]
Mean logloss: 0.6492880073311108


In [16]:
window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

for i, (split, params) in enumerate(zip(splits, parameters)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    X_train = X_train[features[i]]
    X_val = X_val[features[i]]
    X_test = X_test[features[i]]

    X_train = polynomial_features.fit_transform(X_train)
    X_val = polynomial_features.transform(X_val)
    X_test = polynomial_features.transform(X_test)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

# --- Pool everything: concatenate, don't average ---
# Concatenating before scoring naturally weights each window by its
# sample count. Window 6 (252 matches) counts 252x, Window 2 (32) counts
# 32x — no manual weighting needed, unlike averaging six per-window scores.

pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)


# --- Bootstrap CI, so you can tell if an iteration actually moved the needle ---

def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi


mean_ll, lo, hi = bootstrap_logloss_ci(pooled_test_true, pooled_test_proba)
print(f"Pooled log-loss: {mean_ll:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Pooled test log-loss: 0.6545
Pooled test Brier:    0.2319
Pooled test accuracy: 0.5969  (n=516)
{'window': 0, 'n_val': 198, 'n_test': 68, 'val_logloss': 0.6097861497135815, 'test_logloss': 0.7051427194045398, 'test_acc': 0.5294117647058824}
{'window': 1, 'n_val': 132, 'n_test': 32, 'val_logloss': 0.6660186639877167, 'test_logloss': 0.6886593002369626, 'test_acc': 0.5}
{'window': 2, 'n_val': 250, 'n_test': 48, 'val_logloss': 0.6139784761168193, 'test_logloss': 0.5943793761290262, 'test_acc': 0.5833333333333334}
{'window': 3, 'n_val': 252, 'n_test': 68, 'val_logloss': 0.6277509384855506, 'test_logloss': 0.6338541548363482, 'test_acc': 0.6470588235294118}
{'window': 4, 'n_val': 320, 'n_test': 48, 'val_logloss': 0.6317453683044887, 'test_logloss': 0.6445800207840601, 'test_acc': 0.5833333333333334}
{'window': 5, 'n_val': 116, 'n_test': 252, 'val_logloss': 0.6231242819776884, 'test_logloss': 0.6554103694783531, 'test_acc': 0.6190476190476191}
Pooled log-loss: 0.6541  95% CI [0.6218, 0.6866]